# NYC Yellow Taxi 2023 Big Data Analytics and Fare Prediction Project  
## Final Report

---

## 1. Project Overview

This project builds an end-to-end Big Data analytics pipeline using **Databricks, PySpark, Spark SQL, Delta Lake, and Machine Learning**.

The dataset used in this project is the **NYC Yellow Taxi trip data for the full year 2023**. The raw dataset contains **38,310,226 records** stored in monthly Parquet files.

The project follows a structured Big Data workflow starting from raw data ingestion, then data profiling, cleaning, feature engineering, SQL analysis, Gold Layer table creation, visualization planning, and machine learning.

The main goal is to convert raw taxi trip data into clean, reliable, and business-ready insights. The project analyzes taxi demand, passenger charges, trip behavior, airport trips, tipping behavior, speed patterns, pickup zones, route patterns, and financial data quality. Finally, a machine learning model is built to predict the taxi base fare amount.

---

## 2. Business Problem and Objective

Taxi companies, transportation agencies, and city planners need to understand how taxi services operate across time, locations, and trip types.

Raw taxi trip data is very large and may contain missing values, invalid records, extreme outliers, inconsistent financial values, and abnormal trip behavior. Without proper cleaning and validation, analysis results and machine learning models may become misleading.

This project answers business questions such as:

- When is taxi demand highest?
- Which months, days, and hours have the most trips?
- Which pickup zones and routes are most active?
- How do trip distance and duration affect passenger charges?
- How are airport trips different from normal city trips?
- How do passengers tip on credit card trips?
- Can trip features predict the base taxi fare amount?

The objective is to design and implement a scalable Big Data pipeline that ingests, cleans, transforms, analyzes, visualizes, and models NYC Yellow Taxi 2023 trip data.

---

## 3. Tools and Technologies

The project was implemented using:

- Databricks
- PySpark
- Spark SQL
- Delta Lake
- Unity Catalog
- Parquet files
- Machine Learning using PySpark ML

These tools are suitable for Big Data processing because the dataset contains more than **38 million raw records**.

---

## 4. Project Architecture

This project follows the **Medallion Architecture**, which organizes the pipeline into Bronze, Silver, and Gold layers.

---

### Bronze Layer

The Bronze Layer stores the raw taxi data after ingestion.

It contains:

- Raw Parquet data
- Original taxi trip columns
- Source file tracking using `_source_file`
- No major cleaning or transformation

The Bronze table is:

`main.default.yellow_taxi_2023_bronze`

---

### Silver Layer

The Silver Layer contains cleaned and validated taxi trips.

It removes or fixes:

- Duplicate records
- Records outside 2023
- Invalid trip times
- Invalid distances
- Invalid financial values
- Invalid passenger counts
- Invalid rate codes
- Invalid payment types
- Invalid durations
- Unrealistic speeds

The Silver table is:

`main.default.yellow_taxi_2023_silver`

---

### Gold Layer

The Gold Layer contains business-ready analytical tables created using Spark SQL.

These tables summarize the cleaned dataset for dashboards and reporting.

Examples include:

- Monthly demand
- Hourly demand
- Rush hour analysis
- Airport analysis
- Tip analysis
- Distance analysis
- Duration analysis
- Speed analysis
- Top pickup zones
- Top routes
- Financial quality

---

## 5. Dataset Description

The NYC Yellow Taxi dataset contains trip-level information about taxi operations in New York City.

| Column | Description |
|---|---|
| `VendorID` | Taxi technology provider |
| `tpep_pickup_datetime` | Trip pickup date and time |
| `tpep_dropoff_datetime` | Trip dropoff date and time |
| `passenger_count` | Number of passengers |
| `trip_distance` | Trip distance in miles |
| `RatecodeID` | Fare rate type |
| `PULocationID` | Pickup taxi zone ID |
| `DOLocationID` | Dropoff taxi zone ID |
| `payment_type` | Payment method |
| `fare_amount` | Base fare amount |
| `extra` | Extra charges |
| `mta_tax` | MTA tax |
| `tip_amount` | Tip amount |
| `tolls_amount` | Toll charges |
| `improvement_surcharge` | Improvement surcharge |
| `congestion_surcharge` | Congestion surcharge |
| `airport_fee` | Airport-related fee |
| `total_amount` | Total recorded passenger charge |

---

## 6. Step 1 — Data Ingestion into Bronze Layer

In the ingestion step, all 12 monthly Parquet files for 2023 were loaded from the raw data volume.

Each file was read using Spark, selected columns were standardized, and all monthly DataFrames were combined into one Bronze DataFrame.

A `_source_file` column was added to support data lineage and track the original file for each record.

The total number of raw records loaded was:

**38,310,226 rows**

This confirms that the project satisfies the Big Data requirement of using a dataset with more than 3 million rows.

---

## 7. Step 2 — Save Bronze Layer as Delta Table

After ingestion, the Bronze DataFrame was saved as a Delta table in Unity Catalog.

The Bronze table name is:

`main.default.yellow_taxi_2023_bronze`

The table was saved in Delta format to support reliable storage, schema management, and efficient querying.

The Bronze table validation confirmed:

- 38,310,226 rows were saved
- All 12 monthly source files were included
- The schema was successfully stored
- `_source_file` was available for lineage tracking

---

## 8. Step 3 — Bronze Data Profiling

Bronze profiling was performed to understand the raw data quality before cleaning.

The purpose of profiling was to identify:

- What the problem is
- How large the problem is
- How it may affect analysis
- Which Silver cleaning action should be applied

The key principle is:

> Every Silver Layer cleaning rule is based on a profiling result.

---

### 8.1 Source File Coverage

All 12 Parquet files from January to December 2023 were loaded successfully.

The total raw dataset contains:

**38,310,226 rows**

The `_source_file` column is useful for tracking the source file, but it should not be used for time analysis. Time analysis should depend on the actual pickup timestamp.

---

### 8.2 Pickup Date Range

The pickup dates ranged from:

**2001-01-01 to 2024-01-03**

Although the project focuses on 2023, the raw Bronze dataset contains records outside 2023.

This can affect monthly trends and time-based analysis, so the Silver Layer creates `pickup_year` and keeps only records where:

`pickup_year = 2023`

---

### 8.3 Missing Values

Missing value profiling showed that most important columns had no missing values.

However, the following columns had missing values:

- `passenger_count`
- `RatecodeID`
- `store_and_fwd_flag`
- `congestion_surcharge`
- `airport_fee`

Each of these columns had:

- **1,309,356 missing records**
- **3.42% of the dataset**

The same null count appeared across the same group of columns, which suggests the issue is systematic rather than random.

#### Silver Actions

| Column | Silver Action |
|---|---|
| `passenger_count` | Keep only values between 1 and 6 |
| `RatecodeID` | Keep only values between 1 and 6 |
| `store_and_fwd_flag` | Fill nulls with `"Unknown"` |
| `congestion_surcharge` | Fill nulls with 0 |
| `airport_fee` | Fill nulls with 0 |

---

### 8.4 Duplicate Rows

The duplicate check found only:

**2 duplicate rows**

The Silver Layer removes duplicates using `dropDuplicates()`.

---

### 8.5 Temporal Problems

Some trips had dropoff times before pickup times.

Invalid time records:

- **2,475 records**
- **0.00646%**

A taxi trip cannot end before it starts, so the Silver Layer keeps only trips where:

`tpep_dropoff_datetime > tpep_pickup_datetime`

---

### 8.6 Invalid Distance

Distance profiling showed invalid and extreme values.

Invalid distance records included:

- `trip_distance <= 0`
- `trip_distance > 100`

The raw maximum trip distance was extremely high:

**345,729.44 miles**

This is impossible for a taxi trip.

The Silver rule is:

`0 < trip_distance <= 100`

The 100-mile cap is intentionally wide. It keeps rare long trips while removing impossible distance outliers.

---

### 8.7 Detailed Percentile Profiling

Detailed percentile profiling was used to understand the normal range of financial and trip values before applying Silver cleaning rules.

| Field | P99.9 | Raw Max | Silver Rule |
|---|---:|---:|---|
| `fare_amount` | 150 | 386,983.63 | `0 < fare_amount <= 500` |
| `total_amount` | 181.2 | 386,987.63 | `0 < total_amount <= 600` |
| `trip_distance` | 30.05 | 345,729.44 | `0 < trip_distance <= 100` |
| `average_speed_mph` | 48.30 | Over 2.5 million | `0 < average_speed_mph <= 100` |

This proves that the raw data contains extreme outliers. The Silver Layer uses wide business-based caps to remove impossible values while keeping rare but potentially valid trips.

---

### 8.8 Financial Problems

Financial profiling showed invalid, negative, and extreme values in financial columns.

| Quality Check | Records |
|---|---:|
| Invalid fare amount | 394,974 |
| Invalid total amount | 383,193 |
| Negative tip amount | 1,788 |
| Invalid extra | 190,759 |
| Invalid MTA tax | 365,870 |
| Invalid tolls amount | 24,788 |
| Negative improvement surcharge | 376,673 |
| Negative congestion surcharge | 303,052 |
| Negative airport fee | 50,293 |

These values may represent refunds, disputes, corrections, voided trips, or abnormal transactions.

Since the project focuses on normal completed taxi trips, these records are not suitable for the main cleaned dataset.

The Silver financial rules are:

- `0 < fare_amount <= 500`
- `0 < total_amount <= 600`
- `0 <= tip_amount <= 100`
- `0 <= extra <= 20`
- `0 <= mta_tax <= 1`
- `0 <= tolls_amount <= 100`
- `improvement_surcharge >= 0`
- `congestion_surcharge >= 0`
- `airport_fee >= 0`

The project focused first on `fare_amount` and `total_amount` because they are the strongest financial indicators. Then stricter component-level checks were added for better financial data quality.

---

### 8.9 Invalid Passenger Count

Passenger count profiling showed null values, zero passengers, and values greater than 6.

Invalid passenger records:

- **1,892,770 records**
- **4.94%**

The Silver Layer keeps only:

`1 <= passenger_count <= 6`

---

### 8.10 Invalid Payment Type

Payment type profiling showed values such as:

- 0
- 3
- 4
- 5

Normal paid trips are mainly:

- `1 = Credit Card`
- `2 = Cash`

The Silver Layer keeps only:

`payment_type IN (1, 2)`

For tip analysis, only credit card trips are used because cash tips are not reliably recorded.

---

### 8.11 Invalid RatecodeID

Valid `RatecodeID` values are from 1 to 6.

The raw data contained:

- NULL values
- `RatecodeID = 99`

The Silver Layer keeps only:

`1 <= RatecodeID <= 6`

---

### 8.12 Invalid Duration

The raw duration profile showed extreme timestamp problems.

Normal trips had a median duration of:

**12.67 minutes**

But the raw maximum duration was over:

**10,000 minutes**

The Silver Layer keeps only:

`0 < trip_duration_minutes <= 240`

The 240-minute cap is wide enough to keep rare long valid trips while removing extreme timestamp errors.

---

### 8.13 Unrealistic Speed

The raw speed profile showed a median speed of about:

**9.58 mph**

This is realistic for city traffic.

However, the raw maximum speed was over:

**2.5 million mph**

The Silver Layer keeps only:

`0 < average_speed_mph <= 100`

---

### 8.14 Financial Consistency

A financial consistency check was created by comparing `total_amount` with the sum of the financial components.

The result showed:

- **10,227,219 inconsistent records**
- **26.70% of Bronze data**

Because the percentage is large, deleting all inconsistent records would create bias.

Instead, the project uses a flag:

`is_total_amount_inconsistent`

This allows general demand and spatial analysis to use the full cleaned dataset, while strict financial analysis can filter consistent records only.

---

## 9. Step 4 — Silver Layer

The Silver Layer was created from the Bronze table after applying all profiling-based cleaning rules.

The Silver preparation step created helper columns such as:

- `pickup_year`
- `pickup_month`
- `pickup_date`
- `trip_duration_minutes`
- `average_speed_mph`
- `calculated_total_amount`
- `total_amount_difference`
- `is_total_amount_inconsistent`

These columns support filtering, validation, partitioning, and financial quality tracking.

After applying all Silver cleaning rules, the row counts were:

| Metric | Value |
|---|---:|
| Bronze rows | 38,310,226 |
| Silver rows | 35,009,587 |
| Removed rows | 3,300,639 |
| Removed percentage | 8.62% |

The Silver table was saved as:

`main.default.yellow_taxi_2023_silver`

It was partitioned by `pickup_month`.

Silver validation confirmed that all major invalid records were reduced to zero:

- non-2023 records = 0
- invalid time records = 0
- invalid distance records = 0
- invalid fare records = 0
- invalid total records = 0
- invalid passenger records = 0
- invalid ratecode records = 0
- invalid payment records = 0
- invalid duration records = 0
- invalid speed records = 0

This confirms that the Silver Layer is clean, validated, and ready for analysis.

---

## 10. Step 5 — Feature Engineering

Feature engineering was applied after cleaning so that new features were built from trusted Silver data.

The final feature table includes features for:

- Time analysis
- Demand analysis
- Distance analysis
- Duration analysis
- Speed analysis
- Tip behavior
- Payment labels
- Ratecode labels
- Airport trips
- Route analysis
- Financial consistency

| Feature | Purpose |
|---|---|
| `pickup_hour` | Analyze hourly demand |
| `pickup_day_name` | Analyze day-of-week demand |
| `month_name` | Readable month labels |
| `day_type` | Weekday vs weekend |
| `time_period` | Morning, Afternoon, Evening, Night |
| `is_rush_hour` | Rush hour analysis |
| `distance_bucket` | Distance category analysis |
| `duration_bucket` | Duration category analysis |
| `speed_category` | Speed and congestion analysis |
| `tip_base_amount` | Base for tip percentage |
| `tip_percentage` | Credit card tip percentage |
| `tip_category` | Tip behavior classification |
| `payment_type_label` | Readable payment labels |
| `ratecode_label` | Readable rate labels |
| `is_airport_trip` | Airport trip identification |
| `route_pair` | Pickup-to-dropoff route analysis |
| `financial_consistency_label` | Business-readable financial quality label |

The final feature table was saved as:

`main.default.yellow_taxi_2023_features`

It contains:

**35,009,587 rows**

Feature validation showed zero nulls in the main engineered features, confirming that the feature table is ready for SQL analysis, dashboards, and machine learning.

---

## 11. Step 6 — Gold Layer Business SQL Analysis and Visualization

In Step 6, Spark SQL was used to create business-ready Gold tables from the feature table.

The main input table was:

`main.default.yellow_taxi_2023_features`

The Gold tables created were:

- `gold_kpi_summary`
- `gold_monthly_demand`
- `gold_hourly_demand`
- `gold_rush_hour_analysis`
- `gold_day_time_demand`
- `gold_day_of_week_demand`
- `gold_distance_analysis`
- `gold_duration_analysis`
- `gold_airport_analysis`
- `gold_tip_analysis`
- `gold_speed_analysis`
- `gold_top_pickup_zones`
- `gold_top_routes`
- `gold_financial_quality`

---

### 11.1 KPI Summary

The KPI summary provides an executive overview of the cleaned taxi dataset.

| KPI | Value |
|---|---:|
| Total trips | 35,009,587 |
| Total recorded passenger charges | $1,011,580,344.16 |
| Total base fare amount | $687,676,306.04 |
| Average passenger charge | $28.89 |
| Average base fare | $19.64 |
| Average trip distance | 3.49 miles |
| Average duration | 16.30 minutes |
| Average speed | 11.38 mph |
| Credit card trip percentage | 82.62% |
| Cash trip percentage | 17.38% |
| Airport trip percentage | 10.38% |
| Financially consistent percentage | 76.24% |
| Average credit card tip percentage | 17.87% |

The KPI summary shows the large financial and operational scale of NYC Yellow Taxi trips in 2023.

---

### 11.2 Monthly Demand

May had the highest number of trips and the highest total recorded passenger charges.

October was also a strong month.

August and September had lower trip counts, but September had a higher average passenger charge.

**Key insight:**  
May had the most taxi activity overall, while September had fewer trips but higher average value per trip.

---

### 11.3 Hourly Demand

Demand was lowest between late night and early morning, especially around 4 AM.

Demand increased during the day and peaked in the evening, especially around 5 PM to 6 PM.

Early morning trips had higher average passenger charges, likely because trips were longer, faster, or more airport-related.

**Key insight:**  
Evening hours have the highest taxi demand, while early morning trips have lower demand but higher average passenger charges.

---

### 11.4 Rush Hour Analysis

Rush hour trips represented:

**37.10% of trips**

Non-rush hour trips represented:

**62.90% of trips**

Rush hour trips were shorter in distance but slower and slightly longer in duration.

| Period | Average Speed |
|---|---:|
| Rush Hour | 10.60 mph |
| Non-Rush Hour | 11.84 mph |

**Key insight:**  
Rush hour trips are shorter but slower and take slightly longer, showing the effect of traffic congestion.

---

### 11.5 Day and Time Period Analysis

Weekdays represented the majority of taxi demand.

Weekday Afternoon was the busiest time period, with more than 7.56 million trips.

**Key insight:**  
Taxi demand is strongest on weekdays, especially during the afternoon.

---

### 11.6 Day-of-Week Demand

Thursday had the highest demand, followed by Wednesday.

Monday and Sunday had the lowest demand.

**Key insight:**  
Taxi demand is strongest during the middle of the workweek, especially on Wednesday and Thursday.

---

### 11.7 Distance Analysis

Trips under 5 miles represented about **82.82%** of all trips.

The most common category was:

**Short trips between 1 and 2 miles**

Longer trips had higher passenger charges, while very short trips had the highest cost per mile because fixed charges are spread over a very small distance.

**Key insight:**  
Most demand comes from short city trips, while longer trips are fewer but more expensive.

---

### 11.8 Duration Analysis

Almost half of all trips lasted between 5 and 15 minutes.

Trips under 30 minutes represented about 88% of all trips.

**Key insight:**  
Most taxi trips are short trips under 30 minutes.

---

### 11.9 Airport Analysis

Airport trips represented only:

**10.38% of trips**

But they had much higher average passenger charges:

| Trip Type | Average Passenger Charge |
|---|---:|
| Airport Trips | $78.51 |
| Non-Airport Trips | $23.15 |

Airport trips were also longer and faster on average.

**Key insight:**  
Airport trips are fewer, but they are more valuable because they are longer and generate higher passenger charges.

---

### 11.10 Tip Analysis

Tip analysis focused only on credit card trips because cash tips are not reliably recorded.

High Tip was the largest category:

**58.24% of credit card trips**

No Tip trips represented only:

**3.82%**

**Key insight:**  
Most credit card passengers leave tips, and high-tip trips are the dominant category.

---

### 11.11 Speed Analysis

Most trips were normal or slow city trips.

| Speed Category | Percentage |
|---|---:|
| Normal City Speed | 57.10% |
| Very Slow | 33.05% |

This reflects urban traffic conditions.

**Key insight:**  
Most Yellow Taxi trips happen at slow or normal city speeds, which matches congested city driving.

---

### 11.12 Top Pickup Zones

The highest pickup demand came from `PULocationID = 132`.

This zone also had very high passenger charges and long average distance, strongly suggesting airport-related activity.

Other top zones such as 237, 161, 236, and 162 had frequent shorter city trips.

**Key insight:**  
Top pickup zones show two patterns: airport zones generate high-value long trips, while city zones generate frequent short trips.

---

### 11.13 Top Routes

The most common routes were mainly short trips between nearby city zones.

The highest-demand route was:

`237_to_236`

Some routes such as `132_to_230` and `138_to_230` were longer and more expensive, suggesting airport-related travel.

**Key insight:**  
Most common routes are short city routes, while airport-related routes are less frequent but much more valuable per trip.

---

### 11.14 Financial Quality

Financial quality analysis showed:

| Label | Trip Count | Percentage |
|---|---:|---:|
| Consistent | 26,691,136 | 76.24% |
| Inconsistent | 8,318,451 | 23.76% |

The inconsistent records were not deleted because they represent a large share of the dataset. Removing them could create bias in demand, time, airport, and spatial analysis.

Instead, the project uses a flag-based approach.

**Key insight:**  
Financial consistency is tracked transparently, allowing strict financial analysis without damaging general demand analysis.

---

## 12. Step 7 — Machine Learning Model

A machine learning regression model was built using PySpark ML to predict:

`fare_amount`

The model predicts `fare_amount` instead of `total_amount` because `total_amount` includes tips, tolls, taxes, fees, surcharges, and passenger payment behavior.

---

### 12.1 ML Assumption

The model was designed as a **pre-trip fare prediction model**.

Therefore, only features that can reasonably be known before or at the start of the trip were used.

Post-trip or leakage features were excluded, including:

- `total_amount`
- `tip_amount`
- `fare_per_mile`
- `total_per_mile`
- `calculated_total_amount`
- `total_amount_difference`
- `financial_consistency_label`
- `trip_duration_minutes`
- `average_speed_mph`
- `duration_bucket`
- `speed_category`

These features were excluded because they are only known after the trip ends. Using them would make the model look stronger but less realistic.

---

### 12.2 Model Choice

The selected model was:

**Linear Regression**

Linear Regression was chosen because it is:

- Simple
- Fast
- Scalable
- Easy to explain
- Suitable for a Big Data course project
- A strong baseline model for fare prediction

---

### 12.3 Training Process

The model was trained on the full feature dataset, not a small sample.

Dataset size:

**35,009,587 records**

The data was split into:

| Split | Rows |
|---|---:|
| Training set | 28,006,226 |
| Testing set | 7,003,361 |

The selected ML features included:

- `trip_distance`
- `passenger_count`
- `pickup_month`
- `pickup_hour`
- `pickup_day_of_week`
- `is_weekend`
- `is_rush_hour`
- `is_airport_trip`
- `VendorID`
- `RatecodeID`
- `PULocationID`
- `DOLocationID`
- `time_period`
- `day_type`
- `distance_bucket`

Categorical features were processed using:

- `StringIndexer`
- `OneHotEncoder`

All features were combined using:

`VectorAssembler`

Then a Linear Regression model was trained using PySpark ML.

---

### 12.4 Model Evaluation

The model was evaluated using RMSE, MAE, and R².

| Metric | Value |
|---|---:|
| RMSE | 4.46 |
| MAE | 2.38 |
| R² | 0.9377 |

---

### 12.5 Interpretation

The MAE value means that, on average, the model prediction is about:

**$2.38 away from the actual fare**

The R² value means that the model explains about:

**93.77% of the variation in `fare_amount`**

These results are strong for a simple and interpretable pre-trip Linear Regression model.

---

### 12.6 Error Analysis

The largest prediction errors occurred in unusual trips, such as:

- Very high fares for very short distances
- Very low fares for very long distances

These trips are difficult for a simple Linear Regression model because they do not follow the normal fare-distance relationship.

This shows that while the Silver Layer removed clearly invalid records, some unusual but technically valid records can still affect machine learning predictions.

**Key insight:**  
The model performs well overall, but unusual fare-distance patterns create the largest prediction errors.

---

## 13. Visualization Plan

The Gold tables are designed for dashboard visualization.

| Analysis | Recommended Visualization |
|---|---|
| KPI Summary | KPI cards |
| Monthly Demand | Line chart or bar chart |
| Hourly Demand | Line chart |
| Rush Hour Analysis | Bar chart |
| Day-Time Demand | Grouped bar chart or heatmap |
| Day of Week Demand | Bar chart |
| Distance Analysis | Ordered bar chart |
| Duration Analysis | Ordered bar chart |
| Airport Analysis | Bar chart |
| Tip Analysis | Bar chart or donut chart |
| Speed Analysis | Ordered bar chart |
| Top Pickup Zones | Horizontal bar chart |
| Top Routes | Horizontal bar chart |
| Financial Quality | Donut chart or bar chart |
| ML Evaluation | KPI cards for RMSE, MAE, and R² |

---

## 14. Final Conclusion

This project successfully implemented an end-to-end Big Data pipeline for NYC Yellow Taxi 2023 trip data.

The pipeline processed more than **38 million raw records** and produced a cleaned Silver dataset with more than **35 million valid trips**.

The Bronze profiling step revealed several data quality problems, including missing values, timestamp anomalies, invalid distances, financial outliers, invalid passenger counts, invalid payment types, invalid rate codes, unrealistic durations, unrealistic speeds, and financial inconsistencies.

The Silver Layer applied profiling-based cleaning rules to create a reliable dataset for analysis and machine learning.

The Feature Engineering step transformed the cleaned data into business-ready and ML-ready features.

The Gold Layer created SQL-based analytical tables that support demand analysis, passenger charge analysis, tipping behavior, airport trip analysis, speed analysis, zone analysis, route analysis, financial quality analysis, and dashboard visualization.

The Machine Learning step built a Linear Regression model to predict taxi base fare amount using pre-trip features.

The model achieved:

- **RMSE = 4.46**
- **MAE = 2.38**
- **R² = 0.9377**

Overall, the project demonstrates a scalable and complete Big Data solution using Databricks, PySpark, Spark SQL, Delta Lake, and Machine Learning. It converts raw transportation data into clean, reliable, and actionable business insights.